# 02. SBERT + Rule 기반 텍스트 분석

온라인 보험 질문에서 정보 노출 우려와 관련된 후보를 찾기 위해  
SBERT 의미 유사도와 규칙 기반 필터를 함께 사용

In [ ]:
from pathlib import Path
import re

import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


INPUT_PATH = Path("data/aha_medical_insurance_1000.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "jhgan/ko-sroberta-multitask"
SBERT_THRESHOLD = 0.55

In [2]:
df = pd.read_csv(INPUT_PATH)

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()

if "question" not in df.columns:
    df["question"] = (
        df["title"].fillna("")
        + " "
        + df["body"].fillna("")
    ).str.strip()

df["text"] = df["question"].apply(clean_text)
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print("분석 대상:", len(df))

분석 대상: 1000


In [3]:
seed_data = [
    {
        "seed_id": "S1",
        "topic": "회사_단체보험",
        "seed_text": "회사에서 가입한 단체보험으로 보험금을 청구하면 회사가 내가 병원에 다녀온 사실이나 청구 사실을 알 수 있나요?",
    },
    {
        "seed_id": "S2",
        "topic": "우편_가족노출",
        "seed_text": "실손보험을 청구한 뒤 보험사 우편물이 집으로 오면 가족이 청구 사실이나 병원 정보를 알게 될 수 있나요?",
    },
    {
        "seed_id": "S3",
        "topic": "우편_가족노출",
        "seed_text": "보험금 청구 후 집으로 오는 우편물 때문에 가족에게 진료나 청구 사실이 알려질까 걱정된다.",
    },
    {
        "seed_id": "S4",
        "topic": "문자앱_알림노출",
        "seed_text": "실손보험 청구 후 문자나 앱 알림에 병원명이나 청구 내용이 표시되어 다른 사람이 볼 수 있나요?",
    },
    {
        "seed_id": "S5",
        "topic": "문자앱_알림노출",
        "seed_text": "보험금 청구 후 오는 문자나 앱 알림을 가족이 볼까 봐 진료나 청구 사실이 드러날까 걱정된다.",
    },
    {
        "seed_id": "S6",
        "topic": "공동기기_노출",
        "seed_text": "가족과 함께 쓰는 휴대폰이나 태블릿에서 보험 앱을 사용하면 청구내역이나 병원 정보가 보일 수 있나요?",
    },
    {
        "seed_id": "S7",
        "topic": "공동기기_노출",
        "seed_text": "보험 앱을 가족과 같은 기기에서 쓰면 진료내역이나 보험금 청구 사실이 다른 가족에게 노출될 수 있는지 궁금하다.",
    },
    {
        "seed_id": "S8",
        "topic": "가족카드_노출",
        "seed_text": "부모님 카드나 가족카드로 병원비를 결제하면 카드내역의 병원명 때문에 가족이 방문 사실을 알 수 있나요?",
    },
    {
        "seed_id": "S9",
        "topic": "가족카드_노출",
        "seed_text": "가족카드로 병원비를 결제했을 때 카드 명세서를 통해 병원 방문이나 진료 사실이 알려질까 걱정된다.",
    },
    {
        "seed_id": "S10",
        "topic": "가족_대리청구",
        "seed_text": "가족이 내 실손보험 청구를 대신하면 청구 서류에 적힌 병명이나 진료내용을 가족이 볼 수 있나요?",
    },
    {
        "seed_id": "S11",
        "topic": "가족_대리청구",
        "seed_text": "실손보험 청구 서류를 가족이 대신 준비해 줄 때 병원 정보나 진료내용이 가족에게 노출될까 걱정된다.",
    },
]

seed_df = pd.DataFrame(seed_data)
seed_df

,seed_id,topic,seed_text
0,S1,회사_단체보험,회사에서 가입한 단체보험으로 보험금을 청구하면 회사가 내가 병원에 다녀온 사실이나 ...
1,S2,우편_가족노출,실손보험을 청구한 뒤 보험사 우편물이 집으로 오면 가족이 청구 사실이나 병원 정보를...
2,S3,우편_가족노출,보험금 청구 후 집으로 오는 우편물 때문에 가족에게 진료나 청구 사실이 알려질까 걱...
3,S4,문자앱_알림노출,실손보험 청구 후 문자나 앱 알림에 병원명이나 청구 내용이 표시되어 다른 사람이 볼...
4,S5,문자앱_알림노출,보험금 청구 후 오는 문자나 앱 알림을 가족이 볼까 봐 진료나 청구 사실이 드러날까...
5,S6,공동기기_노출,가족과 함께 쓰는 휴대폰이나 태블릿에서 보험 앱을 사용하면 청구내역이나 병원 정보가...
6,S7,공동기기_노출,보험 앱을 가족과 같은 기기에서 쓰면 진료내역이나 보험금 청구 사실이 다른 가족에게...
7,S8,가족카드_노출,부모님 카드나 가족카드로 병원비를 결제하면 카드내역의 병원명 때문에 가족이 방문 사...
8,S9,가족카드_노출,가족카드로 병원비를 결제했을 때 카드 명세서를 통해 병원 방문이나 진료 사실이 알려...
9,S10,가족_대리청구,가족이 내 실손보험 청구를 대신하면 청구 서류에 적힌 병명이나 진료내용을 가족이 볼...


In [4]:
model = SentenceTransformer(MODEL_NAME)

text_embeddings = model.encode(
    df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

seed_embeddings = model.encode(
    seed_df["seed_text"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

similarity_matrix = cosine_similarity(
    text_embeddings,
    seed_embeddings,
)

for i, row in seed_df.iterrows():
    df[f"sim_{row['seed_id']}"] = similarity_matrix[:, i]

sim_cols = [f"sim_{seed_id}" for seed_id in seed_df["seed_id"]]

df["max_similarity"] = df[sim_cols].max(axis=1)
df["best_seed_id"] = (
    df[sim_cols]
    .idxmax(axis=1)
    .str.replace("sim_", "", regex=False)
)

topic_map = dict(zip(seed_df["seed_id"], seed_df["topic"]))
df["best_topic"] = df["best_seed_id"].map(topic_map)

df.to_csv(
    OUTPUT_DIR / "similarity_v2_full_results.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "SBERT >= 0.55:",
    int((df["max_similarity"] >= SBERT_THRESHOLD).sum()),
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT >= 0.55: 468


## Rule filtering

단일 키워드가 아니라 `노출 표현`, `제3자`, `전달 경로`, `민감정보`, `사생활 우려`의 조합으로 후보를 한 번 더 좁힘

In [5]:
exposure_patterns = [
    r"알\s*수\s*있", r"알\s*수가\s*있", r"알게\s*되", r"알게될",
    r"알게\s*될", r"아실까", r"아나요", r"알까요", r"알려지",
    r"알려\s*질", r"알려\s*질까", r"보이", r"볼\s*수\s*있",
    r"볼까", r"보게\s*되", r"확인할\s*수", r"확인\s*가능",
    r"확인하", r"조회", r"열람", r"노출", r"유출", r"드러나",
    r"드러날", r"들키", r"발각", r"표시되", r"표시가\s*되",
    r"뜨나요", r"뜨는지", r"뜰까", r"기록이\s*남", r"기록\s*남",
    r"남나요", r"보내지", r"발송", r"전달되",
]

actor_patterns = [
    r"부모", r"부모님", r"아버지", r"어머니", r"엄마", r"아빠",
    r"가족", r"배우자", r"남편", r"아내", r"자녀", r"아이",
    r"회사", r"직장", r"고용주", r"회사\s*사람", r"보험설계사",
    r"설계사", r"보험\s*설계사", r"다른\s*사람", r"타인",
    r"제3자", r"제\s*3자", r"남에게", r"주변\s*사람",
]

channel_patterns = [
    r"우편", r"우편물", r"안내장", r"문자", r"SMS", r"메시지",
    r"카톡", r"카카오톡", r"앱\s*알림", r"어플\s*알림", r"알림",
    r"푸시", r"명세서", r"카드\s*명세", r"카드\s*내역",
    r"결제\s*내역", r"연말정산", r"휴대폰", r"핸드폰", r"스마트폰",
    r"태블릿", r"공동\s*기기", r"같이\s*쓰는", r"같은\s*기기",
]

sensitive_patterns = [
    r"진료\s*내역", r"진료내역", r"진료\s*기록", r"진료기록",
    r"병원\s*기록", r"병원\s*이력", r"병원이력", r"병원\s*정보",
    r"병원\s*방문", r"병원에\s*다녀", r"병원명", r"병원\s*이름",
    r"진료\s*사실", r"진료사실", r"병명", r"질병\s*이력",
    r"질병이력", r"질병\s*정보", r"정신과", r"산부인과",
    r"청구\s*내역", r"청구내역", r"청구\s*사실",
    r"보험금\s*수령", r"보험금\s*청구", r"수령\s*내역",
    r"개인정보", r"개인\s*정보",
]

privacy_patterns = [
    r"몰래", r"모르게", r"비밀", r"알리고\s*싶지", r"알리고싶지",
    r"알려지는\s*게\s*싫", r"알려질까\s*걱정", r"걱정", r"꺼림칙",
    r"숨기", r"숨길", r"노출될까", r"보일까", r"볼까\s*봐",
    r"알까\s*봐", r"아실까\s*봐",
]


def find_patterns(text, patterns):
    hits = []

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            hits.append(match.group(0))

    return list(dict.fromkeys(hits))

In [6]:
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["analysis_text"] = (
    df["title"] + " " + df["text"]
).str.replace(r"\s+", " ", regex=True).str.strip()

pattern_groups = {
    "exposure": exposure_patterns,
    "actor": actor_patterns,
    "channel": channel_patterns,
    "sensitive": sensitive_patterns,
    "privacy": privacy_patterns,
}

for name, patterns in pattern_groups.items():
    df[f"{name}_hits"] = df["analysis_text"].apply(
        lambda x: find_patterns(x, patterns)
    )
    df[f"has_{name}"] = df[f"{name}_hits"].str.len() > 0

df["sbert_candidate"] = df["max_similarity"] >= SBERT_THRESHOLD

df["rule_score"] = (
    df["has_exposure"].astype(int) * 2
    + df["has_actor"].astype(int) * 2
    + df["has_channel"].astype(int)
    + df["has_sensitive"].astype(int)
    + df["has_privacy"].astype(int)
)

df["rule_exposure_actor"] = df["has_exposure"] & df["has_actor"]
df["rule_exposure_sensitive"] = df["has_exposure"] & df["has_sensitive"]
df["rule_channel_sensitive"] = df["has_channel"] & df["has_sensitive"]
df["rule_privacy"] = (
    df["has_privacy"]
    & (df["has_actor"] | df["has_sensitive"])
)

df["rule_pass"] = (
    df["rule_exposure_actor"]
    | df["rule_exposure_sensitive"]
    | df["rule_channel_sensitive"]
    | df["rule_privacy"]
)

df["hybrid_related"] = (
    df["sbert_candidate"]
    & df["rule_pass"]
)

In [7]:
total_n = len(df)
sbert_n = int(df["sbert_candidate"].sum())
rule_n = int(df["rule_pass"].sum())
hybrid_n = int(df["hybrid_related"].sum())

summary_df = pd.DataFrame({
    "단계": [
        "전체 게시글",
        f"SBERT >= {SBERT_THRESHOLD}",
        "정보노출 Rule 통과",
        "최종 Hybrid 후보",
    ],
    "건수": [
        total_n,
        sbert_n,
        rule_n,
        hybrid_n,
    ],
})

threshold_rows = []
for threshold in [0.50, 0.55, 0.60, 0.65, 0.70]:
    mask = (
        (df["max_similarity"] >= threshold)
        & df["rule_pass"]
    )
    threshold_rows.append({
        "SBERT_threshold": threshold,
        "Hybrid_related_count": int(mask.sum()),
    })

threshold_summary_df = pd.DataFrame(threshold_rows)

display(summary_df)
display(threshold_summary_df)

,단계,건수
0,전체 게시글,1000
1,SBERT >= 0.55,468
2,정보노출 Rule 통과,70
3,최종 Hybrid 후보,53


,SBERT_threshold,Hybrid_related_count
0,0.50,64
1,0.55,53
2,0.60,27
3,0.65,14
4,0.70,4


In [8]:
def get_rule_reason(row):
    reasons = []

    if row["rule_exposure_actor"]:
        reasons.append("노출표현+제3자")
    if row["rule_exposure_sensitive"]:
        reasons.append("노출표현+민감정보")
    if row["rule_channel_sensitive"]:
        reasons.append("전달경로+민감정보")
    if row["rule_privacy"]:
        reasons.append("사생활우려")

    return ", ".join(reasons)


df["rule_reason"] = df.apply(get_rule_reason, axis=1)

related_df = (
    df[df["hybrid_related"]]
    .sort_values(
        ["rule_score", "max_similarity"],
        ascending=[False, False],
    )
    .copy()
)

related_df["manual_label"] = ""
related_df["manual_note"] = ""

df.to_csv(
    OUTPUT_DIR / "hybrid_exposure_full_results.csv",
    index=False,
    encoding="utf-8-sig",
)

related_df.to_csv(
    OUTPUT_DIR / "hybrid_exposure_related_posts.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Hybrid 후보:", len(related_df))
print("검토용 결과는 output 폴더에 저장")

Hybrid 후보: 53
검토용 결과는 output 폴더에 저장


## 결과

| 단계 | 건수 |
|---|---:|
| 전체 게시글 | 1,000 |
| SBERT ≥ 0.55 | 468 |
| 정보노출 Rule 통과 | 70 |
| Hybrid 후보 | 53 |
| 수동 문맥 검토 후 직접 관련 사례 | 13 |

최종 13건은 Hybrid 후보 53건의 원문을 직접 검토해 정리한 결과